# 02 — Preprocessing: Multicollinearity Check (VIF)

**Purpose**: this notebook's first and primary step is a Variance Inflation
Factor (VIF) check across all 20 input features, directly following up on
the EDA finding (`01_EDA.ipynb`) that mean |correlation| across the 210
feature pairs was 0.58, with 17/210 pairs exceeding |r| = 0.7.

Per `IMPLEMENTATION_RULES.md`, multicollinearity must be checked and
reported before any model is considered ready. **This notebook only
computes and reports VIF — no features are dropped, transformed, encoded,
or scaled here.** Those decisions are made explicitly, afterward, once the
numbers have been reviewed.

Research World only (`ml_pipeline/`) — no artifacts are produced by this
notebook.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

DATA_PATH = "../datasets/raw/student_stress_factors.csv"
STRESS_LABEL_COL = "stress_level"
VIF_THRESHOLD = 10.0

## Load dataset and isolate input features

VIF is a property of the *input feature set*, not the target — `stress_level`
is excluded before computing it.

In [2]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=[STRESS_LABEL_COL])
print(f"Input feature matrix: {X.shape[0]} rows x {X.shape[1]} features")
list(X.columns)

Input feature matrix: 1100 rows x 20 features


['anxiety_level',
 'self_esteem',
 'mental_health_history',
 'depression',
 'headache',
 'blood_pressure',
 'sleep_quality',
 'breathing_problem',
 'noise_level',
 'living_conditions',
 'safety',
 'basic_needs',
 'academic_performance',
 'study_load',
 'teacher_student_relationship',
 'future_career_concerns',
 'social_support',
 'peer_pressure',
 'extracurricular_activities',
 'bullying']

## Compute VIF

No `statsmodels` dependency in `mainks` — VIF is implemented directly with
`scikit-learn`, which is already installed, using the standard definition:
for each feature $X_i$, fit an OLS regression of $X_i$ on all other
features, take its $R^2_i$, and compute

$$\text{VIF}_i = \frac{1}{1 - R^2_i}$$

This is mathematically identical to `statsmodels.stats.outliers_influence.
variance_inflation_factor` when the regression includes an intercept (which
`sklearn.linear_model.LinearRegression` does by default).

In [3]:
def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    """Compute Variance Inflation Factor for every column of X."""
    vif_rows = []
    for col in X.columns:
        y_target = X[col]
        X_others = X.drop(columns=[col])
        r2 = LinearRegression().fit(X_others, y_target).score(X_others, y_target)
        vif = np.inf if r2 >= 1.0 else 1.0 / (1.0 - r2)
        vif_rows.append({"feature": col, "r_squared": r2, "VIF": vif})
    return pd.DataFrame(vif_rows).sort_values("VIF", ascending=False).reset_index(drop=True)

vif_table = compute_vif(X)
vif_table

,feature,r_squared,VIF
0,social_support,0.825959,5.745766
1,blood_pressure,0.728752,3.686664
2,future_career_concerns,0.707310,3.416580
3,anxiety_level,0.690055,3.226378
4,self_esteem,0.688315,3.208365
5,teacher_student_relationship,0.687331,3.198274
6,bullying,0.686266,3.187411
7,depression,0.676428,3.090505
8,sleep_quality,0.676279,3.089081
9,safety,0.641595,2.790136


## Features exceeding the VIF < 10 threshold

In [4]:
over_threshold = vif_table[vif_table["VIF"] >= VIF_THRESHOLD].copy()
over_threshold["excess_over_threshold"] = over_threshold["VIF"] - VIF_THRESHOLD

print(f"Threshold: VIF < {VIF_THRESHOLD}")
print(f"Features exceeding threshold: {len(over_threshold)} / {len(vif_table)}")
print()
if len(over_threshold) > 0:
    print(over_threshold.to_string(index=False))
else:
    print("None — all features are under the VIF < 10 threshold.")

Threshold: VIF < 10.0
Features exceeding threshold: 0 / 20

None — all features are under the VIF < 10 threshold.
